# 🇯🇵 RAGScore — 日本語QA生成 & RAG評価デモ

RAGScoreは日本語ドキュメントを自動検出し、日本語でQAペアを生成・評価します。

| 機能 | 説明 |
|------|------|
| 🌍 自動言語検出 | ひらがな・カタカナ・漢字を検出し、日本語プロンプトを使用 |
| 🎯 `--audience` | 対象読者に合わせた質問生成 |
| 📋 `--purpose` | ドキュメントの目的に合わせた質問生成 |

**必要なもの:** `ragscore >= 0.7.5`、OpenAI APIキー（または他の対応プロバイダー）

## 1. インストール

In [ ]:
!pip install -q "ragscore[notebook]" openai numpy

import nest_asyncio
nest_asyncio.apply()
print("✅ 準備完了")

## 2. APIキー設定

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ Colabシークレットからキーを読み込みました")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-..."  # ここにキーを入力
    print("⚠️  ハードコードキーを使用中 — Colabシークレットの使用を推奨")

## 3. 日本語サンプルドキュメント作成

In [ ]:
%%writefile japanese_doc.txt
量子コンピューティングは、量子力学の原理を利用して情報処理を行う革新的な計算技術です。従来のコンピュータがビット（0または1）を使用するのに対し、量子コンピュータは量子ビット（キュービット）を使用します。キュービットは重ね合わせの原理により、0と1の状態を同時に保持することができます。

量子コンピューティングの主要な概念として、重ね合わせ（スーパーポジション）と量子もつれ（エンタングルメント）があります。重ね合わせにより、キュービットは複数の状態を同時に表現でき、並列計算が可能になります。量子もつれは、二つのキュービットが距離に関係なく瞬時に相関する現象で、これにより量子コンピュータは古典的なコンピュータでは不可能な速度で特定の計算を実行できます。

現在、主要な量子コンピューティングプラットフォームとして、IBMのQiskit、GoogleのCirq、MicrosoftのAzure Quantumがあります。IBMは2023年に1,121キュービットのCondorプロセッサを発表し、Googleは2019年に量子超越性を実証しました。量子超越性とは、量子コンピュータが古典的なスーパーコンピュータでは実用的な時間内に解けない問題を解ける能力のことです。

量子コンピューティングの応用分野は多岐にわたります。暗号解読と暗号化（量子暗号通信）、創薬（分子シミュレーション）、金融モデリング（ポートフォリオ最適化）、機械学習（量子機械学習）、サプライチェーン最適化などが主な応用例です。特に、Shorのアルゴリズムは大きな数の素因数分解を効率的に行うことができ、現在のRSA暗号を脅かす可能性があります。

量子コンピューティングの課題として、デコヒーレンス（量子状態の崩壊）、エラー率の高さ、極低温環境（約15ミリケルビン）の必要性があります。量子エラー訂正は活発な研究分野であり、表面コードなどの手法が開発されています。現在の量子コンピュータはNISQ（Noisy Intermediate-Scale Quantum）時代にあり、完全なエラー耐性を持つ量子コンピュータの実現には、まだ数年から数十年かかると予測されています。

日本では、理化学研究所が2023年に国産初の超伝導量子コンピュータを稼働させました。また、富士通は2023年にRIKEN量子コンピュータのクラウドサービスを開始しました。日本政府は量子技術イノベーション戦略を策定し、2030年までに量子技術の社会実装を目指しています。量子コンピューティングの市場規模は、2030年までに約650億ドルに達すると予測されています。

## 4. 言語自動検出テスト

In [ ]:
from ragscore.llm import detect_language

with open("japanese_doc.txt") as f:
    text = f.read()

lang = detect_language(text)
print(f"検出された言語: {lang}")  # 'ja' が表示されるはず
assert lang == "ja", f"期待: 'ja', 検出: '{lang}'"
print("✅ 日本語が正しく検出されました")

## 5. ミニRAGの構築

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()

with open("japanese_doc.txt") as f:
    text = f.read()
chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]
print(f"✅ {len(chunks)}個のチャンクを作成")

def get_embedding(text):
    return client.embeddings.create(input=text, model="text-embedding-3-small").data[0].embedding

chunk_embeddings = np.array([get_embedding(c) for c in chunks])
print(f"✅ {len(chunk_embeddings)}個のチャンクを埋め込み")

def my_rag(question):
    """質問を埋め込み → 上位3チャンクを検索 → GPT-4oで回答"""
    q_emb = np.array(get_embedding(question))
    sims = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    top_idx = np.argsort(sims)[-3:][::-1]
    context = "\n\n".join([chunks[i] for i in top_idx])

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "提供されたコンテキストに基づいてのみ回答してください。簡潔かつ正確に。"},
            {"role": "user", "content": f"コンテキスト:\n{context}\n\n質問: {question}"},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

print("\n🧪 動作確認:")
print(my_rag("量子コンピューティングとは何ですか？"))

## 6. 日本語RAG評価（ベースライン）

In [ ]:
from ragscore import quick_test

result = quick_test(my_rag, docs="japanese_doc.txt", n=5)
print("\n📋 生成された質問:")
for _, row in result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

## 7. 🎯 対象読者：研究者

In [ ]:
researcher_result = quick_test(
    my_rag, docs="japanese_doc.txt", n=5,
    audience="量子コンピューティングの研究者",
    purpose="学術研究",
)
print("\n🔬 研究者向け質問:")
for _, row in researcher_result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

## 8. 🎯 対象読者：ビジネスリーダー

In [ ]:
business_result = quick_test(
    my_rag, docs="japanese_doc.txt", n=5,
    audience="ビジネスリーダーと経営者",
    purpose="投資判断とビジネス戦略",
)
print("\n💼 ビジネスリーダー向け質問:")
for _, row in business_result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

## 9. 結果比較

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "対象読者": ["ベースライン", "研究者", "ビジネスリーダー"],
    "精度": [
        f"{result.accuracy:.0%}",
        f"{researcher_result.accuracy:.0%}",
        f"{business_result.accuracy:.0%}",
    ],
    "平均スコア": [
        f"{result.avg_score:.1f}/5",
        f"{researcher_result.avg_score:.1f}/5",
        f"{business_result.avg_score:.1f}/5",
    ],
    "サンプル質問": [
        result.df.iloc[0]["question"][:60] + "..." if len(result.df) > 0 else "N/A",
        researcher_result.df.iloc[0]["question"][:60] + "..." if len(researcher_result.df) > 0 else "N/A",
        business_result.df.iloc[0]["question"][:60] + "..." if len(business_result.df) > 0 else "N/A",
    ],
})
display(comparison)

---

## 📚 リソース

- **GitHub**: https://github.com/HZYAI/RagScore
- **PyPI**: https://pypi.org/project/ragscore/
- **ウェブサイト**: https://ragscore.io

⭐ RAGScoreが役立つ場合は、GitHubでスターをください！